                     Document RAG

                         Install Required Packages

In [1]:
!pip install langchain langchain-community langchain-core langchain-text-splitters langchain-google-genai chromadb pypdf


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Importing Required Packages

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
import chromadb
import os

1. Indexing

Step 1: Load PDF

In [3]:
import zipfile
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader

# 1. Unzip your files automatically
zip_path = r"C:\Users\Sumathi\OneDrive\Documents\acko-ai-assistant\Policy_Documents-20260519T094537Z-3-001.zip"
extract_to = r"C:\Users\Sumathi\OneDrive\Documents\acko-ai-assistant\extracted_policies"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)
print("Files unzipped successfully!")

# 2. Load all unzipped PDFs at once using PyPDFDirectoryLoader
def Folder_Loader(folder_path):
    loader = PyPDFDirectoryLoader(folder_path)
    return loader.load()

# 3. Call the loader on the newly extracted folder
text_Docs = Folder_Loader(extract_to)

# Check if it worked
print(f"Successfully loaded {len(text_Docs)} pages from your policy PDFs!")

Files unzipped successfully!


Successfully loaded 227 pages from your policy PDFs!


Create Database And Collection

In [4]:
# 1. Install the missing widget utility to stop the background loop freeze
!pip install ipywidgets --quiet

# 2. Shut down the background progress loop system entirely
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Fix applied successfully! Please restart your notebook kernel now.")

Fix applied successfully! Please restart your notebook kernel now.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Step 1: Libraries imported successfully.
Step 10: Triggering function execution...
Step 2: Connecting to ChromaDB at directory: ./chroma_db
Step 3: Client initialized.
Step 5: Deleted old collection 'acko_policies'
Step 6: Setting up Google Embedding model...
Step 7: Embedding model configured.
Step 8: Building Vector Store instance...
Step 9: Vector Store instance built.
🏁 Execution complete!


Text Splitting & Document Injection

In [ ]:
import os
import chromadb
from dotenv import load_dotenv  # Added to read your hidden .env file
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Automatically load environmental keys from your local .env file
load_dotenv()

print("Step 1: Libraries imported successfully.")

# SECURE CONFIGURATION
# Pulling the key dynamically keeps it 100% hidden on GitHub
MY_GEMINI_KEY = os.getenv("GEMINI_KEY")

def Create_Cromadb_Collection(collection_name, dir_path, api_key):
    print(f"Step 2: Connecting to ChromaDB at directory: ./{dir_path}")
    client = chromadb.PersistentClient(path=f'./{dir_path}')
    print("Step 3: Client initialized.")

    existing_collections = [col.name for col in client.list_collections()]
    if collection_name in existing_collections:
        client.delete_collection(collection_name)
        print(f"Step 5: Deleted old collection '{collection_name}'")

    print("Step 6: Setting up Google Embedding model...")
    # FIX: Swapped out the deprecated text-embedding-004 for the active gemini-embedding-001
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001", 
        google_api_key=api_key
    )
    print("Step 7: Embedding model configured.")

    print("Step 8: Building Vector Store instance...")
    Vector_Store = Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embeddings
    )
    print("Step 9: Vector Store instance built.")
    return Vector_Store

print("Step 10: Triggering function execution...")
my_vector_store = Create_Cromadb_Collection("acko_policies", "chroma_db", MY_GEMINI_KEY)
print("🏁 Execution complete!")

Step 1: Libraries imported successfully.
Step 10: Triggering function execution...
Step 2: Connecting to ChromaDB at directory: ./chroma_db
Step 3: Client initialized.
Step 5: Deleted old collection 'acko_policies'
Step 6: Setting up Google Embedding model...
Step 7: Embedding model configured.
Step 8: Building Vector Store instance...
Step 9: Vector Store instance built.
🏁 Execution complete!


In [18]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Slice text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
print("Slicing pages into text chunks...")
chunks = text_splitter.split_documents(text_Docs)
print(f"Created {len(chunks)} searchable text chunks.")

# 2. Ultra-safe slow batch injection
batch_size = 5  # Small batch size to keep the free tier safe
print("Injecting chunks into ChromaDB with strict rate limits...")

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"Uploading batch {i//batch_size + 1}... (Chunks {i} to {i+len(batch)} of {len(chunks)})")
    
    # Upload this specific batch
    my_vector_store.add_documents(batch)
    
    # Take a 7-second break so Google's request counter resets
    if i + batch_size < len(chunks):
        print("Pausing for 7 seconds to respect Google's Free Tier limits... 😴")
        time.sleep(7)

print("🏁 Success! Your full Acko knowledge base is completely indexed and saved!")

Slicing pages into text chunks...
Created 724 searchable text chunks.
Injecting chunks into ChromaDB with strict rate limits...
Uploading batch 1... (Chunks 0 to 5 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 2... (Chunks 5 to 10 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 3... (Chunks 10 to 15 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 4... (Chunks 15 to 20 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 5... (Chunks 20 to 25 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 6... (Chunks 25 to 30 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 7... (Chunks 30 to 35 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits... 😴
Uploading batch 8... (Chunks 35 to 40 of 724)
Pausing for 7 seconds to respect Google's Free Tier limits.

In [2]:
!pip install -U langchain langchain-community langchain-core


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install --force-reinstall langchain langchain-community langchain-core langchain-text-splitters

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain-1.3.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.15-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.8.5-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.16.0-cp313-cp313-win_amd64.whl.metadata (6.5 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using 

In [2]:
!pip install langchain-classic


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


"Testing the AI Chatbot Retrieval (RAG Chain)"

In [ ]:
import os
import time
import chromadb
from dotenv import load_dotenv  # Added to load your hidden .env file
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Load the environmental keys from your local .env file
load_dotenv()

print("🔄 Initializing clean AI components...")

# 1. Secure Credentials and paths
# Pulling the key dynamically keeps it 100% invisible on GitHub
MY_GEMINI_KEY = os.getenv("GEMINI_KEY")
DB_DIR = "chroma_db"
COLLECTION_NAME = "acko_policies"

# Safety verification check
if not MY_GEMINI_KEY:
    raise ValueError("❌ Error: API Key not found! Please check your .env file.")

# 2. Database connection
client = chromadb.PersistentClient(path=f'./{DB_DIR}')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    google_api_key=MY_GEMINI_KEY
)
my_vector_store = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings
)
retriever = my_vector_store.as_retriever(search_kwargs={"k": 4})

# 3. Clean LLM initialization with active production model name
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=MY_GEMINI_KEY,
    temperature=0.3
)

# 4. Prompt construction
system_prompt = (
    "You are a helpful and professional AI insurance assistant for Acko.\n"
    "Answer the user's question using ONLY the provided context below. "
    "If the answer is not present in the context, politely say that you do not "
    "have that information.\n\n"
    "Context:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 5. Build RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
print("✅ AI Chain built successfully with gemini-2.5-flash!")

# 6. Execute Test Question with automated Rate-Limit Handling
test_query = "What is the waiting period for health insurance claims?"
print(f"\nAsking Bot: '{test_query}'...\n")

try:
    response = rag_chain.invoke({"input": test_query})
    print("🤖 Acko Bot Answer:")
    print(response["answer"])
except Exception as e:
    error_str = str(e)
    if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
        print("😴 Quota threshold reached. Waiting 15 seconds for the API counter to clear...")
        time.sleep(15)
        print("🔄 Retrying question now...")
        response = rag_chain.invoke({"input": test_query})
        print("\n🤖 Acko Bot Answer:")
        print(response["answer"])
    else:
        print(f"❌ Error encountered: {e}")

🔄 Initializing clean AI components...
✅ AI Chain built successfully with gemini-2.5-flash!

Asking Bot: 'What is the waiting period for health insurance claims?'...

🤖 Acko Bot Answer:
I do not have information about the waiting period for health insurance claims in the provided context. The context details processing times for various claim types and timelines for grievances.


In [13]:
# A test query about something the bot confirmed it sees in your documents
new_test_query = "What are the processing times for various claim types or timelines for grievances?"

print(f"Asking Bot: '{new_test_query}'...\n")

# Run the query through our successful chain
response = rag_chain.invoke({"input": new_test_query})

print("🤖 Acko Bot Answer:")
print(response["answer"])

Asking Bot: 'What are the processing times for various claim types or timelines for grievances?'...

🤖 Acko Bot Answer:
Here are the processing times for various claim types and timelines for grievances:

*   **Pre-Hospitalisation Reimbursement:** 7–10 working days
*   **Post-Hospitalisation Reimbursement:** 7–10 working days
*   **Grievance on Rejected Claim (Acko's senior team review after submitting representation):** 15 working days
*   **Formal Grievance (email to grievance@acko.com):** Acko must respond within 15 working days.
*   **IRDAI IGMS Escalation (Acko response):** 15 working days

Please note that IRDAI mandates undisputed claims must be settled within 30 days of receiving all required documents.


In [1]:
import psycopg2
import pandas as pd

# Your active Render database link
RENDER_DB_URL = "postgresql://acko_db_user:eAYCpUcQcgzmZJxJZyRBSSCLtOKQXNS0@://render.com"

print("🔄 Connecting to Render Cloud PostgreSQL...")

try:
    # 1. Establish the connection
    conn = psycopg2.connect(RENDER_DB_URL)
    
    # 2. Use Pandas to read the SQL data directly into a beautiful data table!
    query = "SELECT id, created_at, user_question, bot_response FROM customer_chats;"
    df = pd.read_sql_score(query, conn)
    
    # 3. Close connection
    conn.close()
    print("🏁 Connection closed safely.")
    
    # 4. Display the table
    print(f"\n📊 Total Chat Logs Found: {len(df)}")
    display(df)

except Exception as e:
    print(f"❌ Error reading database: {e}")

🔄 Connecting to Render Cloud PostgreSQL...
❌ Error reading database: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

